In [ ]:
# =========================================================
# FORECAST BENCHMARK PROFISSIONAL
# MMP | HOLT-WINTERS | SARIMA | PROPHET | XGBOOST
#
# Parte 1 - Estrutura Base
# =========================================================

# =========================================================
# IMPORTS
# =========================================================

import warnings
warnings.filterwarnings('ignore')

import sqlite3
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# =========================================================
# CONFIGURAÇÕES
# =========================================================

DB_NAME = "gestao_estoque.db"

HORIZONTE_TESTE = 3

SEMENTE = 42

np.random.seed(SEMENTE)

# =========================================================
# LOG
# =========================================================

def log(msg):

    print(
        f"[INFO] {msg}"
    )

# =========================================================
# MÉTRICAS
# =========================================================

def calcular_mae(
    y_true,
    y_pred
):

    return mean_absolute_error(
        y_true,
        y_pred
    )

# ---------------------------------------------------------

def calcular_rmse(
    y_true,
    y_pred
):

    return np.sqrt(

        mean_squared_error(
            y_true,
            y_pred
        )

    )

# ---------------------------------------------------------

def calcular_mape(
    y_true,
    y_pred
):

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mascara = y_true != 0

    if mascara.sum() == 0:

        return 0

    return (

        np.mean(

            np.abs(

                (

                    y_true[mascara]

                    -

                    y_pred[mascara]

                )

                /

                y_true[mascara]

            )

        )

        * 100

    )

# ---------------------------------------------------------

def calcular_metricas(
    y_true,
    y_pred
):

    mae = calcular_mae(
        y_true,
        y_pred
    )

    rmse = calcular_rmse(
        y_true,
        y_pred
    )

    mape = calcular_mape(
        y_true,
        y_pred
    )

    return {

        'MAE': mae,

        'RMSE': rmse,

        'MAPE': mape

    }

# =========================================================
# CONEXÃO SQLITE
# =========================================================

def conectar_banco():

    try:

        conn = sqlite3.connect(
            DB_NAME
        )

        log(
            f'Banco conectado: {DB_NAME}'
        )

        return conn

    except Exception as e:

        raise Exception(

            f'Erro ao conectar: {e}'

        )

# =========================================================
# EXTRAÇÃO DOS DADOS
# =========================================================

def carregar_dados_saida():

    conn = conectar_banco()

    query = """

    SELECT

        DATE(
            data_movimentacao,
            'start of month'
        ) AS mes_ano,

        SUM(quantidade) AS quantidade,

        SUM(valor_total) AS valor_total

    FROM Movimentacoes

    WHERE tipo='S'

    GROUP BY 1

    ORDER BY 1

    """

    df = pd.read_sql_query(
        query,
        conn
    )

    conn.close()

    log(
        f'{len(df)} meses carregados'
    )

    return df

# =========================================================
# CALENDÁRIO CONTÍNUO
# =========================================================

def criar_calendario_continuo(df):

    df['mes_ano'] = pd.to_datetime(
        df['mes_ano']
    )

    calendario = pd.date_range(

        start=df['mes_ano'].min(),

        end=df['mes_ano'].max(),

        freq='MS'

    )

    df = (

        df

        .set_index(
            'mes_ano'
        )

        .reindex(
            calendario
        )

        .fillna(
            0
        )

        .rename_axis(
            'mes_ano'
        )

        .reset_index()

    )

    return df

# =========================================================
# PREPARAÇÃO DA SÉRIE
# =========================================================

def preparar_serie(df):

    serie = (

        df

        .set_index(
            'mes_ano'
        )

        ['quantidade']

        .astype(float)

    )

    return serie

# =========================================================
# TREINO E TESTE
# =========================================================

def separar_treino_teste(
    serie,
    horizonte=3
):

    treino = serie.iloc[
        :-horizonte
    ]

    teste = serie.iloc[
        -horizonte:
    ]

    return treino, teste

# =========================================================
# EXPORTAÇÃO
# =========================================================

def exportar_resultados(
    df_resultado,
    nome_arquivo
):

    df_resultado.to_csv(

        nome_arquivo,

        index=False,

        encoding='utf-8-sig'

    )

    log(

        f'Arquivo salvo: {nome_arquivo}'

    )

# =========================================================
# MAIN TESTE
# =========================================================

if __name__ == '__main__':

    df = carregar_dados_saida()

    df = criar_calendario_continuo(
        df
    )

    serie = preparar_serie(
        df
    )

    treino, teste = separar_treino_teste(
        serie
    )

    print()

    print(
        'Treino:',
        len(treino)
    )

    print(
        'Teste:',
        len(teste)
    )

    print()

    print(
        serie.tail()
    )

[INFO] Banco conectado: gestao_estoque.db
[INFO] 25 meses carregados

Treino: 22
Teste: 3

mes_ano
2026-02-01    14515.0
2026-03-01    20139.0
2026-04-01    19245.0
2026-05-01    20832.0
2026-06-01    20887.0
Name: quantidade, dtype: float64


In [ ]:
# =========================================================
# PARTE 2 - MODELOS
# =========================================================

# =========================================================
# IMPORTS DOS MODELOS
# =========================================================

from statsmodels.tsa.holtwinters import (
    ExponentialSmoothing
)

from statsmodels.tsa.statespace.sarimax import (
    SARIMAX
)

try:

    from prophet import Prophet

    PROPHET_DISPONIVEL = True

except:

    PROPHET_DISPONIVEL = False

try:

    from xgboost import XGBRegressor

    XGB_DISPONIVEL = True

except:

    XGB_DISPONIVEL = False


# =========================================================
# MÉDIA MÓVEL PONDERADA
# =========================================================

def prever_mmp(serie):

    try:

        if len(serie) < 6:

            return float(
                serie.mean()
            )

        pesos = np.array(
            [1, 2, 3, 4, 5, 6]
        )

        pesos = pesos / pesos.sum()

        previsao = np.dot(
            serie.iloc[-6:],
            pesos
        )

        return float(previsao)

    except Exception as e:

        log(
            f'Erro MMP: {e}'
        )

        return float(
            serie.mean()
        )


# =========================================================
# HOLT WINTERS
# =========================================================

def prever_holt_winters(serie):

    try:

        if len(serie) >= 24:

            modelo = ExponentialSmoothing(

                serie,

                trend='add',

                seasonal='add',

                seasonal_periods=12,

                initialization_method='estimated'

            )

        else:

            modelo = ExponentialSmoothing(

                serie,

                trend='add',

                seasonal=None,

                initialization_method='estimated'

            )

        fit = modelo.fit()

        forecast = fit.forecast(1)

        return float(
            forecast.iloc[0]
        )

    except Exception as e:

        log(
            f'Erro Holt-Winters: {e}'
        )

        return float(
            serie.mean()
        )


# =========================================================
# SARIMA
# =========================================================

def prever_sarima(serie):

    try:

        if len(serie) < 36:

            modelo = SARIMAX(

                serie,

                order=(1,1,1),

                seasonal_order=(0,0,0,0),

                enforce_stationarity=False,

                enforce_invertibility=False

            )

        else:

            modelo = SARIMAX(

                serie,

                order=(1,1,1),

                seasonal_order=(1,1,1,12),

                enforce_stationarity=False,

                enforce_invertibility=False

            )

        fit = modelo.fit(
            disp=False
        )

        forecast = fit.forecast(1)

        return float(
            forecast.iloc[0]
        )

    except Exception as e:

        log(
            f'Erro SARIMA: {e}'
        )

        return float(
            serie.mean()
        )


# =========================================================
# PROPHET
# =========================================================

def prever_prophet(serie):

    try:

        if not PROPHET_DISPONIVEL:

            return float(
                serie.mean()
            )

        prophet_df = pd.DataFrame({

            'ds': pd.date_range(

                start='2024-01-01',

                periods=len(serie),

                freq='MS'

            ),

            'y': serie.values

        })

        modelo = Prophet(

            yearly_seasonality=True,

            weekly_seasonality=False,

            daily_seasonality=False

        )

        modelo.fit(
            prophet_df
        )

        future = modelo.make_future_dataframe(

            periods=1,

            freq='MS'

        )

        forecast = modelo.predict(
            future
        )

        return float(

            forecast[
                'yhat'
            ].iloc[-1]

        )

    except Exception as e:

        log(
            f'Erro Prophet: {e}'
        )

        return float(
            serie.mean()
        )


# =========================================================
# XGBOOST
# =========================================================

def prever_xgboost(serie):

    try:

        if not XGB_DISPONIVEL:

            return float(
                serie.mean()
            )

        if len(serie) < 8:

            return float(
                serie.mean()
            )

        df = pd.DataFrame({

            'y': serie

        })

        df['lag1'] = df['y'].shift(1)

        df['lag2'] = df['y'].shift(2)

        df['lag3'] = df['y'].shift(3)

        df['roll_mean_3'] = (

            df['y']

            .rolling(3)

            .mean()

        )

        df['roll_std_3'] = (

            df['y']

            .rolling(3)

            .std()

        )

        df = df.dropna()

        X = df[

            [

                'lag1',

                'lag2',

                'lag3',

                'roll_mean_3',

                'roll_std_3'

            ]

        ]

        y = df['y']

        modelo = XGBRegressor(

            n_estimators=200,

            learning_rate=0.05,

            max_depth=3,

            objective='reg:squarederror',

            random_state=SEMENTE

        )

        modelo.fit(
            X,
            y
        )

        ultimo = pd.DataFrame({

            'lag1': [serie.iloc[-1]],

            'lag2': [serie.iloc[-2]],

            'lag3': [serie.iloc[-3]],

            'roll_mean_3': [

                serie.iloc[-3:].mean()

            ],

            'roll_std_3': [

                serie.iloc[-3:].std()

            ]

        })

        forecast = modelo.predict(
            ultimo
        )

        return float(
            forecast[0]
        )

    except Exception as e:

        log(
            f'Erro XGBoost: {e}'
        )

        return float(
            serie.mean()
        )


# =========================================================
# DICIONÁRIO DOS MODELOS
# =========================================================

MODELOS = {

    'MMP': prever_mmp,

    'HOLT_WINTERS': prever_holt_winters,

    'SARIMA': prever_sarima,

    'PROPHET': prever_prophet,

    'XGBOOST': prever_xgboost

}


# =========================================================
# TESTE DOS MODELOS
# =========================================================

if __name__ == '__main__':

    df = carregar_dados_saida()

    df = criar_calendario_continuo(df)

    serie = preparar_serie(df)

    print()

    print('=' * 60)

    print('TESTE DOS MODELOS')

    print('=' * 60)

    print()

    for nome, func in MODELOS.items():

        forecast = func(serie)

        print(

            f'{nome:<15} '

            f'-> '

            f'{forecast:,.2f}'

        )

[INFO] Banco conectado: gestao_estoque.db
[INFO] 25 meses carregados

TESTE DOS MODELOS

MMP             -> 19,709.00
HOLT_WINTERS    -> 21,431.87
SARIMA          -> 20,010.76


INFO:prophet:n_changepoints greater than number of observations. Using 19.


PROPHET         -> 22,149.37
XGBOOST         -> 19,566.27


In [ ]:
# =========================================================
# PARTE 3 - BENCHMARK
# =========================================================

# =========================================================
# WALK FORWARD VALIDATION
# =========================================================

def walk_forward_validation(
    serie,
    modelo_func,
    horizonte=3
):

    treino = serie.iloc[:-horizonte]

    teste = serie.iloc[-horizonte:]

    historico = treino.copy()

    previsoes = []

    for valor_real in teste.values:

        forecast = modelo_func(
            historico
        )

        previsoes.append(
            forecast
        )

        historico = pd.concat([

            historico,

            pd.Series(
                [valor_real]
            )

        ], ignore_index=True)

    return {

        'real': teste.values,

        'previsto': previsoes

    }

# =========================================================
# AVALIAR MODELO
# =========================================================

def avaliar_modelo(
    serie,
    nome_modelo,
    modelo_func
):

    try:

        resultado = walk_forward_validation(

            serie,

            modelo_func,

            horizonte=HORIZONTE_TESTE

        )

        metricas = calcular_metricas(

            resultado['real'],

            resultado['previsto']

        )

        return {

            'Modelo': nome_modelo,

            'MAE': metricas['MAE'],

            'RMSE': metricas['RMSE'],

            'MAPE': metricas['MAPE']

        }

    except Exception as e:

        log(

            f'Erro avaliação {nome_modelo}: {e}'

        )

        return {

            'Modelo': nome_modelo,

            'MAE': np.nan,

            'RMSE': np.nan,

            'MAPE': np.nan

        }

# =========================================================
# EXECUTAR BENCHMARK
# =========================================================

def executar_benchmark(serie):

    resultados = []

    print()

    print('=' * 60)

    print('EXECUTANDO BENCHMARK')

    print('=' * 60)

    print()

    for nome, func in MODELOS.items():

        print(

            f'Executando {nome}...'

        )

        resultado = avaliar_modelo(

            serie,

            nome,

            func

        )

        resultados.append(
            resultado
        )

    df_resultados = pd.DataFrame(
        resultados
    )

    return df_resultados

# =========================================================
# RANKING
# =========================================================

def gerar_ranking(
    df_resultados
):

    ranking = (

        df_resultados

        .sort_values(
            by='MAPE',
            ascending=True
        )

        .reset_index(
            drop=True
        )

    )

    ranking.index += 1

    return ranking

# =========================================================
# MELHOR MODELO
# =========================================================

def selecionar_melhor_modelo(
    ranking
):

    return ranking.iloc[0]['Modelo']

# =========================================================
# FORECAST FINAL
# =========================================================

def forecast_final(
    serie,
    melhor_modelo
):

    func = MODELOS[
        melhor_modelo
    ]

    previsao = func(
        serie
    )

    return previsao

# =========================================================
# EXPORTAÇÃO
# =========================================================

def exportar_benchmark(
    ranking
):

    ranking.to_csv(

        'benchmark_modelos.csv',

        index=True,

        encoding='utf-8-sig'

    )

    log(

        'benchmark_modelos.csv exportado'

    )

# =========================================================
# TABELA FORMATADA
# =========================================================

def exibir_resultados(
    ranking
):

    print()

    print('=' * 60)

    print('RANKING DOS MODELOS')

    print('=' * 60)

    print()

    print(

        ranking.round(2)

    )

# =========================================================
# EXECUÇÃO
# =========================================================

if __name__ == '__main__':

    df = carregar_dados_saida()

    df = criar_calendario_continuo(
        df
    )

    serie = preparar_serie(
        df
    )

    ranking = executar_benchmark(
        serie
    )

    ranking = gerar_ranking(
        ranking
    )

    exibir_resultados(
        ranking
    )

    exportar_benchmark(
        ranking
    )

    melhor_modelo = selecionar_melhor_modelo(
        ranking
    )

    previsao = forecast_final(

        serie,

        melhor_modelo

    )

    print()

    print('=' * 60)

    print('MELHOR MODELO')

    print('=' * 60)

    print()

    print(

        f'Modelo: {melhor_modelo}'

    )

    print(

        f'Forecast Próximo Mês: {previsao:,.2f}'

    )

[INFO] Banco conectado: gestao_estoque.db
[INFO] 25 meses carregados

EXECUTANDO BENCHMARK

Executando MMP...
Executando HOLT_WINTERS...
Executando SARIMA...


INFO:prophet:n_changepoints greater than number of observations. Using 16.


Executando PROPHET...


INFO:prophet:n_changepoints greater than number of observations. Using 17.
INFO:prophet:n_changepoints greater than number of observations. Using 18.


Executando XGBOOST...

RANKING DOS MODELOS

         Modelo      MAE     RMSE   MAPE
1        SARIMA   966.01  1083.45   4.68
2           MMP  1221.08  1241.96   6.05
3  HOLT_WINTERS  1341.21  1486.72   6.64
4       XGBOOST  2573.09  2758.19  12.53
5       PROPHET  6669.09  7319.82  33.33
[INFO] benchmark_modelos.csv exportado

MELHOR MODELO

Modelo: SARIMA
Forecast Próximo Mês: 20,010.76


In [ ]:
# =========================================================
# PARTE 3 - BENCHMARK
# =========================================================

# =========================================================
# WALK FORWARD VALIDATION
# =========================================================

def walk_forward_validation(
    serie,
    modelo_func,
    horizonte=3
):

    treino = serie.iloc[:-horizonte]

    teste = serie.iloc[-horizonte:]

    historico = treino.copy()

    previsoes = []

    for valor_real in teste.values:

        forecast = modelo_func(
            historico
        )

        previsoes.append(
            forecast
        )

        historico = pd.concat([

            historico,

            pd.Series(
                [valor_real]
            )

        ], ignore_index=True)

    return {

        'real': teste.values,

        'previsto': previsoes

    }

# =========================================================
# AVALIAR MODELO
# =========================================================

def avaliar_modelo(
    serie,
    nome_modelo,
    modelo_func
):

    try:

        resultado = walk_forward_validation(

            serie,

            modelo_func,

            horizonte=HORIZONTE_TESTE

        )

        metricas = calcular_metricas(

            resultado['real'],

            resultado['previsto']

        )

        return {

            'Modelo': nome_modelo,

            'MAE': metricas['MAE'],

            'RMSE': metricas['RMSE'],

            'MAPE': metricas['MAPE']

        }

    except Exception as e:

        log(

            f'Erro avaliação {nome_modelo}: {e}'

        )

        return {

            'Modelo': nome_modelo,

            'MAE': np.nan,

            'RMSE': np.nan,

            'MAPE': np.nan

        }

# =========================================================
# EXECUTAR BENCHMARK
# =========================================================

def executar_benchmark(serie):

    resultados = []

    print()

    print('=' * 60)

    print('EXECUTANDO BENCHMARK')

    print('=' * 60)

    print()

    for nome, func in MODELOS.items():

        print(

            f'Executando {nome}...'

        )

        resultado = avaliar_modelo(

            serie,

            nome,

            func

        )

        resultados.append(
            resultado
        )

    df_resultados = pd.DataFrame(
        resultados
    )

    return df_resultados

# =========================================================
# RANKING
# =========================================================

def gerar_ranking(
    df_resultados
):

    ranking = (

        df_resultados

        .sort_values(
            by='MAPE',
            ascending=True
        )

        .reset_index(
            drop=True
        )

    )

    ranking.index += 1

    return ranking

# =========================================================
# MELHOR MODELO
# =========================================================

def selecionar_melhor_modelo(
    ranking
):

    return ranking.iloc[0]['Modelo']

# =========================================================
# FORECAST FINAL
# =========================================================

def forecast_final(
    serie,
    melhor_modelo
):

    func = MODELOS[
        melhor_modelo
    ]

    previsao = func(
        serie
    )

    return previsao

# =========================================================
# EXPORTAÇÃO
# =========================================================

def exportar_benchmark(
    ranking
):

    ranking.to_csv(

        'benchmark_modelos.csv',

        index=True,

        encoding='utf-8-sig'

    )

    log(

        'benchmark_modelos.csv exportado'

    )

# =========================================================
# TABELA FORMATADA
# =========================================================

def exibir_resultados(
    ranking
):

    print()

    print('=' * 60)

    print('RANKING DOS MODELOS')

    print('=' * 60)

    print()

    print(

        ranking.round(2)

    )

# =========================================================
# EXECUÇÃO
# =========================================================

if __name__ == '__main__':

    df = carregar_dados_saida()

    df = criar_calendario_continuo(
        df
    )

    serie = preparar_serie(
        df
    )

    ranking = executar_benchmark(
        serie
    )

    ranking = gerar_ranking(
        ranking
    )

    exibir_resultados(
        ranking
    )

    exportar_benchmark(
        ranking
    )

    melhor_modelo = selecionar_melhor_modelo(
        ranking
    )

    previsao = forecast_final(

        serie,

        melhor_modelo

    )

    print()

    print('=' * 60)

    print('MELHOR MODELO')

    print('=' * 60)

    print()

    print(

        f'Modelo: {melhor_modelo}'

    )

    print(

        f'Forecast Próximo Mês: {previsao:,.2f}'

    )

[INFO] Banco conectado: gestao_estoque.db
[INFO] 25 meses carregados

EXECUTANDO BENCHMARK

Executando MMP...
Executando HOLT_WINTERS...
Executando SARIMA...
Executando PROPHET...


INFO:prophet:n_changepoints greater than number of observations. Using 16.
INFO:prophet:n_changepoints greater than number of observations. Using 17.
INFO:prophet:n_changepoints greater than number of observations. Using 18.


Executando XGBOOST...

RANKING DOS MODELOS

         Modelo      MAE     RMSE   MAPE
1        SARIMA   966.01  1083.45   4.68
2           MMP  1221.08  1241.96   6.05
3  HOLT_WINTERS  1341.21  1486.72   6.64
4       XGBOOST  2573.09  2758.19  12.53
5       PROPHET  6669.09  7319.82  33.33
[INFO] benchmark_modelos.csv exportado

MELHOR MODELO

Modelo: SARIMA
Forecast Próximo Mês: 20,010.76
